# Observer advantage, Part II: Mistral-7B and Qwen2.5-7B on a Colab A100

Runs the registered Part II cells (`docs/PREREG_observer_advantage_keys.md`) for the two
long-context Tier A models. Amendment 2 of that registration assigns them here: every cell of a
model runs on one GPU product (A100) with Atlas's package versions, and Llama-2 and Tier B stay on
Atlas.

**How to use.** Runtime -> Change runtime type -> **A100 GPU**. Then Runtime -> Run all.
Results go to Google Drive, `MyDrive/tqp-keys/<model>/<arm>/`. After a disconnect, just Run all
again: finished cells are skipped and an unfinished task resumes at the next document.

Nothing here reads or decides a result; the scorer runs later, on all models together.

In [ ]:
# 1. The GPU must be an A100 for every session (a model's cells may not mix GPUs).
import subprocess
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip()
print(gpu)
assert "A100" in gpu, "Runtime -> Change runtime type -> A100 GPU"

In [ ]:
# 2. Google Drive: every output is written here, so a disconnect loses nothing.
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/tqp-keys"
import os
os.makedirs(ROOT, exist_ok=True)
print(ROOT, sorted(os.listdir(ROOT)))

In [ ]:
# 3. Atlas's package versions (Amendment 2). About 3 minutes.
!pip -q install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip -q install transformers==5.5.0 tokenizers==0.22.2 accelerate==1.13.0 datasets==4.3.0 \
    huggingface_hub==1.11.0 safetensors==0.7.0 sentencepiece==0.2.1 numpy==2.2.6 \
    rouge==1.0.1 jieba==0.42.1 fuzzywuzzy==0.18.0 python-Levenshtein
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# 4. The registered harness (tag keys-colab-v1), LongBench's config and metrics (the commit
#    Atlas uses), and the three task files.
TAG = "keys-colab-v1"
!rm -rf /content/tqp && git clone -q --depth 1 --branch {TAG} https://github.com/ahb-sjsu/turboquant-pro /content/tqp
!git -C /content/tqp log --oneline -1
!test -d /content/LongBench || (git clone -q https://github.com/THUDM/LongBench /content/LongBench && git -C /content/LongBench checkout -q 2e00731)
!test -f /content/lb_data/data/qasper.jsonl || (mkdir -p /content/lb_data && cd /content/lb_data && \
    wget -q https://huggingface.co/datasets/THUDM/LongBench/resolve/main/data.zip && \
    unzip -q -o data.zip "data/trec.jsonl" "data/triviaqa.jsonl" "data/qasper.jsonl")
!wc -l /content/lb_data/data/*.jsonl

## Run

First every arm a registered verdict reads (the identity-codebook gates and `fp16` first), then
every reported arm. Progress is appended to `MyDrive/tqp-keys/run.log`; cell outputs stream below.

In [ ]:
# 5. The run. Safe to interrupt and re-run at any time.
import os, subprocess
env = {**os.environ, "HF_HOME": "/content/hf", "TOKENIZERS_PARALLELISM": "false",
       "PYTHONPATH": "/content/tqp:/content/tqp/benchmarks"}
KV = "/content/tqp/benchmarks/kvquant_matrix"
common = ["--root", ROOT, "--models", "mistral-7b-instruct,qwen2.5-7b-instruct", "--gpu", "0",
          "--lbroot", "/content/LongBench/LongBench", "--datadir", "/content/lb_data/data"]
for arms in ("priority", "all"):
    subprocess.run(["python", f"{KV}/keys_run.py", *common, "--arms", arms], env=env, cwd=KV, check=True)
print("ALL CELLS DONE")

In [ ]:
# 6. Status (run any time, e.g. in a second session or after a disconnect).
!tail -n 25 /content/drive/MyDrive/tqp-keys/run.log